In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""---"""
# look at neurons that have that block side difference and see where they land on the scatters

# TAVG

In [ ]:
import numpy as np
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id, norm=True)
encoder.verify()

encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()

encoder_mf.verify()
encoder_mf.fit_encoder()

## r2

In [ ]:
# r2 between DMS and DLS
from core.viz import plot_kdes
from utils.colors import colors_region

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}
# scores
scores = {
    k: {
        f"{reg}, {model}": encoder_.scores[model][encoder_.reg_idxs[reg]]
        for reg in encoder_.regions
        for model in ["baseline", "encoder"]
    }
    for k, encoder_ in encoders.items()
}

# styles
linestyles = {"baseline": "--", "encoder": "-"}
linewidths = {"baseline": 0.5, "encoder": 1}
styles = {
    f"{reg}, {model}": {
        "linestyle": linestyles[model],
        "linewidth": linewidths[model],
        "color": colors_region[reg],
    }
    for reg in encoder.regions
    for model in ["baseline", "encoder"]
}

for k, scores_ in scores.items():
    plot_kdes(
        scores_,
        label=rf"$r^2$, {k}",
        xlim=(-0.25, 1),
        add_means=False,
        line_kwargs=styles,
    )

In [ ]:
from utils.colors import colors_strategy


def plot_r2_reg(reg="DMS"):
    scores_encoder = {
        k: encoder_.scores["encoder"][encoder_.reg_idxs[reg]]
        for k, encoder_ in encoders.items()
    }
    styles = {
        "full": {"color": "#444444", "linewidth": 1},
        "mb": {
            "color": colors_strategy["mb"],
        },
        "mf": {
            "color": colors_strategy["mf"],
        },
    }
    plot_kdes(
        scores_encoder,
        xlim=[-0.5, 1.0],
        label=rf"$r^2$, encoder, {reg}",
        add_means=False,
        line_kwargs=styles,
    )


plot_r2_reg(reg="DMS")
plot_r2_reg(reg="DLS")

## weight comp between regions and strategies

### kde

In [ ]:
from core.viz import plot_kde_row

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}

sc_mean = {
    k: {
        reg: encoder_.robs[:, encoder_.reg_idxs[reg]].mean(axis=0)
        for reg in encoder_.regions
    }
    for k, encoder_ in encoders.items()
}

plot_kde_row(sc_mean)

In [ ]:
from core.viz import plot_kde_row
from utils.colors import colors_region
from utils.paths import FIGURES_DIR

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}

# styles
linewidths_strategy = {"full": 0.5, "mb": 1, "mf": 1}

styles_strategy = {f"{reg}": {"color": colors_region[reg]} for reg in encoder.regions}
styles_reg = {
    f"{k}": {
        "color": colors_strategy[k],
        "linewidth": linewidths_strategy[k],
    }
    for k in encoders
}

# iterate through all regressors and save
for regressor in encoder.dm_names:
    if "tents" not in regressor:
        # weights
        weights_strategy = {
            k: {
                f"{reg}": encoder_.encoder_weights[
                    encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                ]
                for reg in encoder_.regions
            }
            for k, encoder_ in encoders.items()
        }

        weights_reg = {
            reg: {
                f"{k}": encoder_.encoder_weights[
                    encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                ]
                for k, encoder_ in encoders.items()
            }
            for reg in encoder.regions
        }

        # plot
        for k, weights, styles in zip(
            ["strategy", "region"],
            [weights_strategy, weights_reg],
            [styles_strategy, styles_reg],
        ):
            fig, _ = plot_kde_row(
                weights, styles, title=rf"$\beta$ {regressor}", add_means=False
            )

            fpath = FIGURES_DIR / "lite" / subj_id / sess_id / "bweight"
            fpath.mkdir(parents=True, exist_ok=True)
            fig.savefig(fpath / f"{regressor}_{k}.png", dpi=300, bbox_inches="tight")

### scatter, hist 2d, contour

In [ ]:
from core.data import tv_vals
from core.viz import plot_scatter, plot_hist2d, plot_contour, plot_2d_row
from utils.viz_utils import save_fig

fpath = FIGURES_DIR / "lite" / subj_id / sess_id / "bweight"

for regr in encoder.tv_keys:
    vals = [tv_vals[regr][0]] if not regr == "response_prev" else tv_vals[regr]

    for val in vals:
        regressor = f"{regr}_{val}"
        weights = {
            reg: [
                encoder_.encoder_weights[
                    encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                ]
                for encoder_ in [encoder_mb, encoder_mf]
            ]
            for reg in encoder.regions
        }

        # scatter
        fig, ax = plot_2d_row(
            plot_scatter,
            weights,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            add_unity=True,
            add_lr=True,
        )

        save_fig(fig, fpath / "scatter", f"{regressor}.png")

        # mn/mx
        mn = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).min()
        )
        mx = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).max()
        )

        # hist2d
        fig, ax = plot_2d_row(
            plot_hist2d,
            weights,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            sharey=True,
            sharex=True,
        )

        save_fig(fig, fpath / "hist2d", f"{regressor}.png")

        # contour
        fig, ax = plot_2d_row(
            plot_contour,
            weights,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            fill=False,
            sharey=True,
            sharex=True,
        )

        save_fig(fig, fpath / "contour", f"{regressor}.png")

# TR

## r2

### single session

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"
stepsize_s = 0.25

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=stepsize_s,
)

encoder_mb = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=stepsize_s,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=stepsize_s,
    strategy_filter="mf",
)

encoder.fit_encoder()
encoder_mb.fit_encoder()
encoder_mf.fit_encoder()

In [ ]:
# control
from core.data import get_strategy_filter_idxs

idxs = get_strategy_filter_idxs(encoder.trial_data)

idxs_mb = np.sort(
    np.random.choice(np.concatenate((idxs["mb"], idxs["mf"])), len(idxs["mb"]))
)
idxs_mf = np.sort(
    np.random.choice(np.concatenate((idxs["mb"], idxs["mf"])), len(idxs["mb"]))
)

encoder_mb_ctrl = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    idxs=idxs_mb,
    stepsize_s=stepsize_s,
    strategy_filter="mb",
)

encoder_mf_ctrl = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    idxs=idxs_mf,
    stepsize_s=stepsize_s,
    strategy_filter="mf",
)

encoder_mb_ctrl.fit_encoder()
encoder_mf_ctrl.fit_encoder()

In [ ]:
(encoder_mb_ctrl.trial_data.strategy == 1).mean()

In [ ]:
import numpy as np
from scipy.stats import pearsonr as r

rs = {
    regr: {
        reg: np.array(
            [
                r(
                    encoder_mb.encoder_weights[
                        encoder_mb.reg_idxs[reg], encoder_mb.dm_idxs[f"{regr}_{i}"]
                    ],
                    encoder_mf.encoder_weights[
                        encoder_mf.reg_idxs[reg], encoder_mf.dm_idxs[f"{regr}_{i}"]
                    ],
                ).statistic
                for i in range(encoder.num_bins)
            ]
        )
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
}

In [ ]:
from utils.colors import colors_region

fig, axes = plt.subplots(
    ncols=len(encoder.tv_keys), nrows=1, figsize=(6, 2), sharey=True, tight_layout=True
)

for i, regr in enumerate(encoder.tv_keys):
    ax = axes[i]

    for reg in encoder.regions:
        ax.plot(rs[regr][reg], c=colors_region[reg], label=reg)
    ax.axhline(y=0, color="#444444")
    ax.legend()

### plot all tpoints together

In [ ]:
from core.data import tv_vals, subject_ids, session_ids
from core.viz import plot_scatter, plot_2d_row
from sg.models import make_tre, StrategyEncoder
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR
import numpy as np

for subj_id in ["MR82", "MR83"]:
    print(subj_id)
    fpath = FIGURES_DIR / "time_resolved" / "bweight" / subj_id / "dme"

    weights_master = {
        reg: {
            strategy: {regr: [] for regr in tv_vals.keys() if regr != "strategy"}
            for strategy in ["mb", "mf"]
        }
        for reg in ["DMS", "DLS"]
    }

    for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
        print(sess_id)
        try:
            encoder_mb = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                norm=True,
                stepsize_s=0.1,
                strategy_filter="mb",
            )

            encoder_mf = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                norm=True,
                stepsize_s=0.1,
                strategy_filter="mf",
            )

            encoder_mb.fit_encoder()
            encoder_mf.fit_encoder()

        except ValueError:
            print("..too few trials")
            continue

        for reg, weights_reg in weights_master.items():
            for regr in weights_reg["mb"].keys():
                try:
                    idx0, idxm1 = (
                        encoder_mb.dm_idxs[f"{regr}_0"],
                        encoder_mb.dm_idxs[f"{regr}_{encoder_mb.num_bins - 1}"],
                    )
                    weights_reg["mb"][regr].extend(
                        np.concatenate(
                            encoder_mb.encoder_weights[
                                encoder_mb.reg_idxs[reg], idx0 : idxm1 + 1
                            ]
                        )
                    )
                    weights_reg["mf"][regr].extend(
                        np.concatenate(
                            encoder_mf.encoder_weights[
                                encoder_mf.reg_idxs[reg], idx0 : idxm1 + 1
                            ]
                        )
                    )
                except KeyError:
                    continue

    for regressor in encoder_mb.tv_keys:
        weights = {
            reg: np.array(
                [
                    weights_master[reg]["mb"][regressor],
                    weights_master[reg]["mf"][regressor],
                ]
            )
            for reg in ["DMS", "DLS"]
        }

        # mn/mx
        mn = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).min()
        )
        mx = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).max()
        )

        # scatter
        fig, ax = plot_2d_row(
            plot_scatter,
            weights,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            add_unity=True,
            add_lr=True,
        )

        save_fig(fig, fpath / "scatter", f"{regressor}.png")

In [ ]:
# TODO
# - oopify
# - control idxs
# - bootstrap

### plot tpoints separately

In [ ]:
from core.data import get_strategy_filter_idxs


def get_bweights_all(subj_id, stepsize_s=0.25, control=False, **kwargs):
    sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]

    encoder = make_tre(Encoder, tr_type="dme")(
        subj_id,
        sess_ids[0],
        norm=True,
        stepsize_s=stepsize_s,
    )
    encoder.get_data()

    weights_master = {
        reg: {
            strategy: {
                regr: {i: [] for i in range(encoder.num_bins)}
                for regr in encoder.tv_keys
            }
            for strategy in ["mb", "mf"]
        }
        for reg in encoder.regions
    }

    def _build_encoder(strategy, sess_id):
        if control:
            encoder = Encoder(
                subj_id,
                sess_id,
            )
            encoder.get_data()

            # get an even percentage of model-based and model-free
            idxs = get_strategy_filter_idxs(
                encoder.trial_data, balance_strategy=True, cond_balance=False
            )
            idxs = np.sort(
                np.random.choice(
                    np.concatenate((idxs["mb"], idxs["mf"])), len(idxs["mb"])
                )
            )

            encoder_strategy = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                stepsize_s=stepsize_s,
                idxs=idxs,
                strategy_filter=strategy,
                **kwargs,
            )
            encoder_strategy.fit_encoder()

            assert len(np.unique(encoder_strategy.trial_data["strategy"])) == 2

            print(strategy, (encoder_strategy.trial_data["strategy"] == 1).mean())

        else:
            encoder_strategy = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                stepsize_s=stepsize_s,
                strategy_filter=strategy,
                **kwargs,
            )
            encoder_strategy.fit_encoder()
            assert len(np.unique(encoder_strategy.trial_data["strategy"])) == 1

        return encoder_strategy

    for sess_id in sess_ids:
        print(sess_id)
        try:
            encoder_mb = _build_encoder(strategy="mb", sess_id=sess_id)
            encoder_mf = _build_encoder(strategy="mf", sess_id=sess_id)

        except ValueError:
            print("..too few trials")
            continue

        for reg, weights_reg in weights_master.items():
            for regr in weights_reg["mb"].keys():
                for i in range(encoder.num_bins):
                    try:
                        weights_reg["mb"][regr][i].extend(
                            encoder_mb.encoder_weights[
                                encoder_mb.reg_idxs[reg],
                                encoder_mb.dm_idxs[f"{regr}_{i}"],
                            ]
                        )
                        weights_reg["mf"][regr][i].extend(
                            encoder_mf.encoder_weights[
                                encoder_mf.reg_idxs[reg],
                                encoder_mf.dm_idxs[f"{regr}_{i}"],
                            ]
                        )
                    except KeyError:
                        continue
    return weights_master, encoder

In [ ]:
weights_master_ctrl, _ = get_bweights_all(
    subj_id="MR82", stepsize_s=0.1, norm=True, control=True
)

In [ ]:
weights_master, encoder = get_bweights_all(
    subj_id="MR82", stepsize_s=0.1, norm=True, control=False
)

### get correlation

In [ ]:
# pearson r
from scipy.stats import pearsonr as r


def get_bweight_strategy_r(weights_master, encoder):
    rs_master = {
        reg: {regr: np.zeros(encoder.num_bins) for regr in encoder.tv_keys}
        for reg in encoder.regions
    }

    for reg in encoder.regions:
        for regressor in encoder.tv_keys:
            weights_mb = weights_master[reg]["mb"][regressor]
            weights_mf = weights_master[reg]["mf"][regressor]

            rs_master[reg][regressor] = np.array(
                [
                    r(weights_mb[i], weights_mf[i]).statistic
                    for i in range(encoder.num_bins)
                ]
            )
    return rs_master

In [ ]:
rs_master = get_bweight_strategy_r(weights_master, encoder)
rs_master_ctrl = get_bweight_strategy_r(weights_master_ctrl, encoder)

In [ ]:
# linear regression slope
from sklearn.linear_model import LinearRegression as LR


def get_bweight_strategy_m(weights_master, encoder):
    ms_master = {
        reg: {regr: np.zeros(encoder.num_bins) for regr in encoder.tv_keys}
        for reg in encoder.regions
    }

    for reg in encoder.regions:
        for regressor in encoder.tv_keys:
            weights_mb = weights_master[reg]["mb"][regressor]
            weights_mf = weights_master[reg]["mf"][regressor]

            ms = np.zeros(encoder.num_bins)

            for i in range(encoder.num_bins):
                lr = LR().fit(np.array(weights_mb[i]).reshape(-1, 1), weights_mf[i])
                ms[i] = lr.coef_[0]
            ms_master[reg][regressor] = ms
    return ms_master

In [ ]:
ms_master = get_bweight_strategy_m(weights_master, encoder)
ms_master_ctrl = get_bweight_strategy_m(weights_master_ctrl, encoder)

### plot correlation

In [ ]:
from utils.colors import colors_regressor, colors_region


def plot_bweight_strategy_corr(metrics, encoder, metric="", mode="regr"):
    """
    expecting metrics to be structured like:
        {reg: {regr: {i: metric}}}
    """

    def _accessorize(ax):
        ax.axvline(x=0, linestyle="--", linewidth=0.75, color="#666666")
        ax.axvline(x=0.5, linestyle="--", linewidth=0.5, color="#666666")
        ax.axhline(y=0, linewidth=1, color="k")
        ax.legend(loc="upper left")

        ax.set_xlabel("Time (s)")
        ax.set_ylabel(metric)
        ax.set_title(reg)

    # regressors together, regions apart
    if mode == "regr":
        _, axes = plt.subplots(nrows=1, ncols=2, figsize=(7, 3), sharey=True)

        for i, reg in enumerate(encoder.regions):
            for regr in encoder.tv_keys:
                axes[i].plot(
                    encoder_mb.tbin_centers,
                    metrics[reg][regr],
                    color=colors_regressor[regr],
                    label=regr,
                )
        _accessorize(axes[i])

    # regions apart, regressors together
    elif mode == "reg":
        fig, axes = plt.subplots(
            nrows=1, ncols=len(encoder.tv_keys), figsize=(10, 2), sharey=True
        )

        for i, regr in enumerate(encoder.tv_keys):
            for reg in encoder.regions:
                axes[i].plot(
                    encoder.tbin_centers,
                    metrics[reg][regr],
                    color=colors_region[reg],
                    label=reg,
                )

            _accessorize(axes[i])
    else:
        raise ValueError("valid arguments for mode are 'regr' and 'reg'")

    return axes

In [ ]:
plot_bweight_strategy_corr(
    metrics=rs_master, metric=r"$r$", encoder=encoder, mode="reg"
)

In [ ]:
plot_bweight_strategy_corr(
    metrics=rs_master_ctrl, metric=r"$r$", encoder=encoder, mode="reg"
)

### r vs m

In [ ]:
from core.viz import plot_scatter

rs_master_flat = np.concatenate(
    [rs_master[reg][regr] for reg in ["DMS", "DLS"] for regr in encoder.tv_keys]
)
ms_master_flat = np.concatenate(
    [ms_master[reg][regr] for reg in ["DMS", "DLS"] for regr in encoder.tv_keys]
)

plot_scatter(
    rs_master_flat,
    ms_master_flat,
    xlabel=r"$r$",
    ylabel="slope",
    add_lr=True,
    add_unity=True,
)

### archive

In [ ]:
for subj_id in ["MR82", "MR83"]:
    for regressor in encoder_mb.tv_keys:
        fig, axes = plt.subplots(
            ncols=2,
            nrows=encoder.num_bins,
            figsize=(
                2 * 2,
                encoder.num_bins * 2,
            ),
            tight_layout=True,
            sharey=True,
            sharex=True,
        )

        weights_ = np.concatenate(
            [
                weights_master[subj_id][reg][strategy][regressor][i]
                for reg in ["DMS", "DLS"]
                for strategy in ["mb", "mf"]
                for i in range(encoder.num_bins)
            ]
        )
        mn = np.min(weights_)
        mx = np.max(weights_)

        for i in range(encoder.num_bins):
            weights = {
                reg: np.array(
                    [
                        weights_master[subj_id][reg]["mb"][regressor][i],
                        weights_master[subj_id][reg]["mf"][regressor][i],
                    ]
                )
                for reg in ["DMS", "DLS"]
            }

            _, _ = plot_2d_row(
                plot_scatter,
                weights,
                mn=mn,
                mx=mx,
                xlabel="mb",
                ylabel="mf",
                title=rf"$\beta$ {regressor}",
                add_unity=True,
                add_lr=True,
                fig=fig,
                axes=axes[i],
            )

        save_fig(fig, fpath / "scatter", f"{regressor}_t.png")

In [ ]:
for regressor in encoder_mb.tv_keys:
    fig, axes = plt.subplots(
        ncols=2,
        nrows=encoder.num_bins,
        figsize=(
            2 * 2,
            encoder.num_bins * 2,
        ),
        tight_layout=True,
        sharey=True,
        sharex=True,
    )

    weights_ = np.concatenate(
        [
            weights_master[reg][strategy][regressor][i]
            for reg in ["DMS", "DLS"]
            for strategy in ["mb", "mf"]
            for i in range(encoder.num_bins)
        ]
    )
    mn = np.min(weights_)
    mx = np.max(weights_)

    for i in range(encoder.num_bins):
        weights = {
            reg: np.array(
                [
                    weights_master[reg]["mb"][regressor][i],
                    weights_master[reg]["mf"][regressor][i],
                ]
            )
            for reg in ["DMS", "DLS"]
        }

        _, _ = plot_2d_row(
            plot_scatter,
            weights,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            add_unity=True,
            add_lr=True,
            fig=fig,
            axes=axes[i],
        )

save_fig(fig, fpath / "scatter", f"{regressor}_t.png")

In [ ]:
# separate by time point

for subj_id in ["MR82", "MR83"]:
    print(subj_id)
    fpath = FIGURES_DIR / "time_resolved" / "bweight" / subj_id / "dme"

    for i in range(encoder.num_bins):
        weights_master = {
            reg: {
                strategy: {regr: [] for regr in tv_vals.keys() if regr != "strategy"}
                for strategy in ["mb", "mf"]
            }
            for reg in ["DMS", "DLS"]
        }

        for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
            print(sess_id)
            try:
                encoder_mb = make_tre(StrategyEncoder)(
                    subj_id,
                    sess_id,
                    norm=True,
                    stepsize_s=0.25,
                    strategy_filter="mb",
                )

                encoder_mf = make_tre(StrategyEncoder)(
                    subj_id,
                    sess_id,
                    norm=True,
                    stepsize_s=0.25,
                    strategy_filter="mf",
                )

                encoder_mb.fit_encoder()
                encoder_mf.fit_encoder()

            except ValueError:
                print("..too few trials")
                continue

            for reg, weights_reg in weights_master.items():
                for regr in weights_reg["mb"].keys():
                    try:
                        weights_reg["mb"][regr].extend(
                            # np.concatenate(
                            encoder_mb.encoder_weights[
                                encoder_mb.reg_idxs[reg],
                                encoder_mb.dm_idxs[f"{regr}_{i}"],
                            ]
                            # )
                        )
                        weights_reg["mf"][regr].extend(
                            # np.concatenate(
                            encoder_mf.encoder_weights[
                                encoder_mf.reg_idxs[reg],
                                encoder_mf.dm_idxs[f"{regr}_{i}"],
                            ]
                            # )
                        )
                    except KeyError:
                        continue

        for regressor in encoder_mb.tv_keys:
            weights = {
                reg: np.array(
                    [
                        weights_master[reg]["mb"][regressor],
                        weights_master[reg]["mf"][regressor],
                    ]
                )
                for reg in ["DMS", "DLS"]
            }

            # mn/mx
            mn = (
                1.05
                * np.concatenate(
                    [
                        weights_
                        for weights_strategy in weights.values()
                        for weights_ in weights_strategy
                    ]
                ).min()
            )
            mx = (
                1.05
                * np.concatenate(
                    [
                        weights_
                        for weights_strategy in weights.values()
                        for weights_ in weights_strategy
                    ]
                ).max()
            )

            # scatter
            fig, ax = plot_2d_row(
                plot_scatter,
                weights,
                mn=mn,
                mx=mx,
                xlabel="mb",
                ylabel="mf",
                title=rf"$\beta$ {regressor}, t={i}",
                add_unity=True,
                add_lr=True,
            )

            save_fig(fig, fpath / "scatter", f"{regressor}_t{i}.png")

### diff

In [ ]:
# abs diff

ds_master = {
    subj_id: {
        reg: {regr: np.zeros(encoder.num_bins) for regr in encoder.tv_keys}
        for reg in ["DMS", "DLS"]
    }
    for subj_id in ["MR82", "MR83"]
}

for subj_id in ["MR82", "MR83"]:
    for reg in ["DMS", "DLS"]:
        for regressor in encoder.tv_keys:
            weights_mb = weights_master[subj_id][reg]["mb"][regressor]
            weights_mf = weights_master[subj_id][reg]["mf"][regressor]

            ds_master[subj_id][reg][regressor] = np.array(
                [
                    np.abs(weights_mb[i]) - np.abs(weights_mf[i])
                    for i in range(encoder.num_bins)
                ]
            )

In [ ]:
# for subj_id in ["MR82", "MR83"]
from core.viz import plot_kdes

# time together, regions apart
fig, axes = plt.subplots(
    nrows=len(encoder.tv_keys),
    ncols=2,
    figsize=(8, 6),
    tight_layout=True,
    sharey=True,
    sharex=True,
)

for i, reg in enumerate(["DMS", "DLS"]):
    for j, regr in enumerate(encoder.tv_keys):
        ax = axes[j, i]

        diffs = {t: ds_master["MR82"][reg][regr][t] for t in range(encoder.num_bins)}

        for t in range(encoder.num_bins):
            plot_kdes(diffs, ax=ax, legend=False)

        ax.set_title(f"{reg}, {regr}")

In [ ]:
from utils.colors import colors_region

# regressors apart, regions together
fig, axes = plt.subplots(
    nrows=1, ncols=len(encoder.tv_keys), figsize=(10, 2), sharey=True
)

for i, regr in enumerate(encoder.tv_keys):
    for reg in ["DMS", "DLS"]:
        axes[i].plot(
            encoder_mb.tbin_centers,
            ms_master["MR82"][reg][regr],
            color=colors_region[reg],
            label=reg,
        )

    axes[i].axvline(x=0, linestyle="--", linewidth=0.75, color="#666666")
    axes[i].axvline(x=0.5, linestyle="--", linewidth=0.5, color="#666666")
    axes[i].axhline(y=0, linewidth=1, color="k")
    axes[i].legend(loc="upper left")

    axes[i].set_xlabel("Time (s)")
    axes[i].set_ylabel("slope")
    axes[i].set_title(regr)

## weight comp between regions and strategies

### kde

In [ ]:
from core.viz import plot_kde_row

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}

sc_mean = {
    k: {
        f"{reg}": encoder_.robs[:, :, encoder_.reg_idxs[reg]].mean(axis=(0, 1))
        * (1000 / encoder.binwidth_ms)
        for reg in encoder_.regions
    }
    for k, encoder_ in encoders.items()
}

styles_reg = {reg: styles[f"{reg}, encoder"] for reg in encoder.regions}
_ = plot_kde_row(
    sc_mean, line_kwargs=styles_reg, label="avg fr (across trials & tbins)"
)

In [ ]:
encoder.fit_encoder()

In [ ]:
encoder.fit_encoder()
encoder_mb.fit_encoder()
encoder_mf.fit_encoder()

In [ ]:
regressor = "response_right"
weights_strategy = {
    k: {
        f"{reg}": encoder_.encoder_weights[
            encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
        ]
        for reg in encoder_.regions
    }
    for k, encoder_ in encoders.items()
}

In [ ]:
from core.viz import plot_kde_row
from utils.colors import colors_region, colors_strategy
from utils.paths import FIGURES_DIR

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}

# styles
linewidths_strategy = {"full": 0.5, "mb": 1, "mf": 1}

styles_strategy = {f"{reg}": {"color": colors_region[reg]} for reg in encoder.regions}
styles_reg = {
    f"{k}": {
        "color": colors_strategy[k],
        "linewidth": linewidths_strategy[k],
    }
    for k in encoders
}

# iterate through all regressors and save
for regressor in encoder.dm_names:
    if "tents" not in regressor:
        # weights
        weights_strategy = {
            k: {
                f"{reg}": np.ravel(
                    encoder_.encoder_weights[
                        :, encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                    ]
                )
                for reg in encoder_.regions
            }
            for k, encoder_ in encoders.items()
        }

        weights_reg = {
            reg: {
                f"{k}": np.ravel(
                    encoder_.encoder_weights[
                        :, encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                    ]
                )
                for k, encoder_ in encoders.items()
            }
            for reg in encoder.regions
        }

        # plot
        for k, weights, styles in zip(
            ["strategy", "region"],
            [weights_strategy, weights_reg],
            [styles_strategy, styles_reg],
        ):
            fig, _ = plot_kde_row(
                weights, styles, title=rf"$\beta$ {regressor}", add_means=False
            )

            fpath = FIGURES_DIR / "time_resolved" / "bweight" / subj_id / sess_id
            fpath.mkdir(parents=True, exist_ok=True)
            fig.savefig(fpath / f"{regressor}_{k}.png", dpi=300, bbox_inches="tight")

### scatter, hist 2d, contour

In [ ]:
from core.data import tv_vals
from core.viz import plot_scatter, plot_2d_row
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

for regr in encoder.tv_keys:
    norm_str = "norm" if encoder.norm else "nonorm"
    fpath = FIGURES_DIR / "time_resolved" / "bweight" / norm_str

    vals = [tv_vals[regr][0]] if not regr == "response_prev" else tv_vals[regr]

    for val in vals:
        regressor = f"{regr}_{val}"
        print(regressor)
        weights = {
            reg: [
                np.concatenate(
                    encoder_.encoder_weights[
                        :, encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                    ]
                )
                for encoder_ in [encoder_mb, encoder_mf]
            ]
            for reg in encoder.regions
        }

        idxs = {
            reg: np.tile(range(encoder.num_bins), encoder.psths[reg].shape[0])
            for reg in encoder.regions
        }

        # mn/mx
        mn = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).min()
        )
        mx = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).max()
        )

        mn = -3
        mx = 3

        # scatter
        fig, ax = plot_2d_row(
            plot_scatter,
            weights,
            # color=idxs,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            add_unity=True,
            add_lr=True,
        )

        save_fig(
            fig,
            fpath / "scatter" / regressor / subj_id,
            f"{regressor}-{subj_id}_{sess_id}.png",
        )

In [ ]:
encoder.norm

In [ ]:
len(weights["DLS"][1])

## across sessions

In [ ]:
from core.data import tv_vals

tv_vals

In [ ]:
from core.data import tv_vals, subject_ids, session_ids
from core.viz import plot_scatter, plot_2d_row
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

for subj_id in ["MR82", "MR83"]:
    print(subj_id)
    fpath = FIGURES_DIR / "time_resolved" / "bweight" / subj_id

    weights_master = {
        reg: {
            strategy: {
                f"{regr}_{val}": []
                for regr, vals in tv_vals.items()
                for val in vals
                if regr != "strategy"
            }
            for strategy in ["mb", "mf"]
        }
        for reg in ["DMS", "DLS"]
    }

    for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
        print(sess_id)
        try:
            encoder_mb = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                stepsize_s=0.1,
                strategy_filter="mb",
            )

            encoder_mf = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                stepsize_s=0.1,
                strategy_filter="mf",
            )

            encoder_mb.fit_encoder()
            encoder_mf.fit_encoder()

        except ValueError:
            print("..too few trials")
            continue

        for reg, weights_reg in weights_master.items():
            for k in weights_reg["mb"].keys():
                print(k)
                try:
                    weights_reg["mb"][k].extend(
                        np.concatenate(
                            encoder_mb.encoder_weights[
                                :, encoder_mb.reg_idxs[reg], encoder_mb.dm_idxs[k]
                            ]
                        )
                    )
                    weights_reg["mf"][k].extend(
                        np.concatenate(
                            encoder_mf.encoder_weights[
                                :, encoder_mf.reg_idxs[reg], encoder_mf.dm_idxs[k]
                            ]
                        )
                    )
                except KeyError:
                    continue

    for regr in encoder.tv_keys:
        vals = [tv_vals[regr][0]] if not regr == "response_prev" else tv_vals[regr]

        for val in vals:
            regressor = f"{regr}_{val}"
            weights = {
                reg: [weights_master[reg]["mb"], weights_master[reg]["mf"]]
                for reg in ["DMS", "DLS"]
            }

            # mn/mx
            mn = (
                1.05
                * np.concatenate(
                    [
                        weights_
                        for weights_strategy in weights.values()
                        for weights_ in weights_strategy
                    ]
                ).min()
            )
            mx = (
                1.05
                * np.concatenate(
                    [
                        weights_
                        for weights_strategy in weights.values()
                        for weights_ in weights_strategy
                    ]
                ).max()
            )

            # scatter
            fig, ax = plot_2d_row(
                plot_scatter,
                weights,
                mn=mn,
                mx=mx,
                xlabel="mb",
                ylabel="mf",
                title=rf"$\beta$ {regressor}",
                add_unity=True,
                add_lr=True,
            )

            save_fig(fig, fpath / "scatter", f"{regressor}.png")